# Create an interface with strain matching

Use Zur and McGill superlattices matching [algorithm](https://doi.org/10.1063/1.3330840) to select an interface between two materials with minimal strain.

<h2 style="color:green">Usage</h2>

1. Make sure to select Input Materials (in the outer runtime) before running the notebook.
1. Set notebook parameters in cell 1.1. below (or use the default values).
1. Set slab parameters for the substrate and film in cell 2.4. (or use default).
1. Click “Run” > “Run All” to run all cells.
1. Wait for the run to complete (depending on the parameters can take a few min). 
1. Scroll down to view results. 

## Summary
1. Prepare the Environment: Set up the notebook and install packages, preview the input materials
1. Create substrate and film slabs and select the terminations
1. Generate interfaces with strain matcher and plot strain vs number of atoms 
1. Select the interface with the desired strain and visualize it

## Notes
1. We perform strain matching on the slabs to extract the supercell dimensions. The algorithm has a set of parameters, such as the maximum area considered.
1. When the strain matching is finished, the interface with the lowest strain (and the smallest number of atoms) is selected. 
1. ZSL strain matching is performed using Pymatgen [implementation](https://pymatgen.org/pymatgen.analysis.interfaces.html#pymatgen.analysis.interfaces.zsl).
1. For more information, see [Introduction](Introduction.ipynb)
<!-- # TODO: use a hashtag-based anchor link to interface creation documention above -->


## 1. Prepare the Environment
### 1.1. Set up the notebook 

Set the following flags to control the notebook behavior 

In [ ]:
FILM_INDEX = 1  # Index in the list of materials, to access as materials[FILM_INDEX]
FILM_MILLER_INDICES = (0, 0, 1)
FILM_THICKNESS = 1  # in atomic layers
FILM_TERMINATION_FORMULA = None  # if None, the first termination will be used
FILM_VACUUM = 0.0  # in angstroms
FILM_XY_SUPERCELL_MATRIX = [[1, 0], [0, 1]]
FILM_USE_ORTHOGONAL_C = True

SUBSTRATE_INDEX = 0
SUBSTRATE_MILLER_INDICES = (0, 0, 1)
SUBSTRATE_THICKNESS = 3  # in atomic layers
SUBSTRATE_TERMINATION_FORMULA = None  # if None, the first termination will be used
SUBSTRATE_VACUUM = 0.0  # in angstroms
SUBSTRATE_XY_SUPERCELL_MATRIX = [[1, 0], [0, 1]]
SUBSTRATE_USE_ORTHOGONAL_C = True

INTERFACE_DISTANCE = 3.0  # Gap between substrate and film, in Angstrom. If None, calculated automatically using RDF
INTERFACE_VACUUM = 10.0  # Vacuum over film, in Angstrom
CENTER_IN_Z = True  # Centre the interface along z in the cell

# Whether to convert materials to conventional cells before creating slabs.
# To create interfaces with smaller cells, set this flag to False. (and pass already conventional cells as input)
USE_CONVENTIONAL_CELL = True

# Maximum area for the superlattice search algorithm (the final interface area will be smaller)
MAX_AREA = 150  # in Angstrom^2
# Additional fine-tuning parameters (increase values to get more strained matches):
MAX_AREA_TOLERANCE = 0.09  # in Angstrom^2
MAX_LENGTH_TOLERANCE = 0.05
MAX_ANGLE_TOLERANCE = 0.02

# Whether to reduce the resulting interface cell to the primitive cell after the interface creation.
# If the reduction causes unexpected results, try increasing the `MAX_AREA` for search.
REDUCE_RESULT_CELL_TO_PRIMITIVE = True

### 1.2. Install Packages
The step executes only in Pyodide environment. For other environments, the packages should be installed via `pip install` (see [README](../../README.ipynb)).

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made")

### 1.3. Get input materials and assign `substrate` and `film`
Materials are loaded with `get_data()`. The first material is assigned as substrate and the second as film.

In [ ]:
from mat3ra.notebooks_utils.material import get_materials

materials = get_materials(globals())
substrate = materials[SUBSTRATE_INDEX]
try:
    film = materials[FILM_INDEX]
except IndexError:
    print("Film material not found. Re-using substrate material as film.")
    film = substrate

### 1.4. Preview Substrate and Film

In [ ]:
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials as visualize

visualize([substrate, film], repetitions=[3, 3, 1], rotation="0x")

## 2. Configure slabs for interface


### 2.1. Get possible terminations for the slabs

In [ ]:
from mat3ra.made.tools.helpers import get_slab_terminations

film_slab_terminations = get_slab_terminations(material=film, miller_indices=FILM_MILLER_INDICES)
substrate_slab_terminations = get_slab_terminations(material=substrate, miller_indices=SUBSTRATE_MILLER_INDICES)
print("Film slab terminations:", film_slab_terminations)
print("Substrate slab terminations:", substrate_slab_terminations)

### 2.2. Visualize slabs for all possible terminations

In [ ]:
from mat3ra.made.tools.helpers import create_slab, select_slab_termination
from mat3ra.made.tools.helpers import create_interface_zsl_between_slabs as create_zsl_interface_between_slabs

film_slabs = [create_slab(film, miller_indices=FILM_MILLER_INDICES, termination_top=termination, vacuum=0) for
              termination
              in
              film_slab_terminations]

substrate_slabs = [
    create_slab(substrate, miller_indices=SUBSTRATE_MILLER_INDICES, termination_top=termination, vacuum=0)
    for termination in
    substrate_slab_terminations]

film_slabs_with_titles = [{"material": slab, "title": str(termination)} for slab, termination in
                          zip(film_slabs, film_slab_terminations)]
substrate_slabs_with_titles = [{"material": slab, "title": str(termination)} for slab, termination in
                               zip(substrate_slabs, substrate_slab_terminations)]

visualize(film_slabs_with_titles, repetitions=[3, 3, 1], rotation="-90x")
visualize(substrate_slabs_with_titles, repetitions=[3, 3, 1], rotation="-90x")

### 2.3. Select terminations for the Slabs

In [ ]:
film_termination = select_slab_termination(film_slab_terminations, FILM_TERMINATION_FORMULA)
substrate_termination = select_slab_termination(substrate_slab_terminations, SUBSTRATE_TERMINATION_FORMULA)

### 2.4. Create Substrate and Layer Slabs
Slab Configuration lets define the slab thickness, vacuum, and the Miller indices of the interfacial plane and get the slabs with possible terminations.
Define the substrate slab cell that will be used as a base for the interface and the film slab cell that will be placed on top of the substrate slab.

In [ ]:
from mat3ra.made.tools.build.pristine_structures.two_dimensional.slab import SlabConfiguration, SlabBuilder

substrate_slab_config = SlabConfiguration.from_parameters(
    material_or_dict=substrate,
    miller_indices=SUBSTRATE_MILLER_INDICES,
    number_of_layers=SUBSTRATE_THICKNESS,
    vacuum=0.0,
    termination_top_formula=SUBSTRATE_TERMINATION_FORMULA,
    use_conventional_cell=USE_CONVENTIONAL_CELL
)

film_slab_config = SlabConfiguration.from_parameters(
    material_or_dict=film,
    miller_indices=FILM_MILLER_INDICES,
    number_of_layers=FILM_THICKNESS,
    vacuum=0.0,
    termination_bottom_formula=FILM_TERMINATION_FORMULA,
    use_conventional_cell=USE_CONVENTIONAL_CELL
)

substrate_slab = SlabBuilder().get_material(substrate_slab_config)
film_slab = SlabBuilder().get_material(film_slab_config)

### 2.5. Calculate Interfacial Distance from RDF (if not specified)
If INTERFACE_DISTANCE is set to None, we automatically calculate it based on the Radial Distribution Function (RDF) of the bulk materials. 

The algorithm:
1. Creates temporary 3x3x3 supercells of substrate and film bulk materials to capture full atomic repetition
2. Calculates RDF for each supercell to find the first peak (nearest neighbor distance)
3. Averages the two first-peak distances to estimate the interfacial distance

This provides a reasonable initial guess based on the atomic structure of the materials.

In [ ]:
from mat3ra.made.tools.analyze.interface import calculate_interfacial_distance_from_rdf

if INTERFACE_DISTANCE is None:
    INTERFACE_DISTANCE = calculate_interfacial_distance_from_rdf(
        substrate_material=substrate,
        film_material=film,
        rdf_cutoff=10.0,
        rdf_bin_size=0.1,
        supercell_size=(3, 3, 3)
    )
    print(f"Using RDF-based interfacial distance: {INTERFACE_DISTANCE:.3f} Angstrom")
else:
    print(f"Using specified interfacial distance: {INTERFACE_DISTANCE:.3f} Angstrom")

### 2.6. Analyze possible interfaces with ZSL Analyzer
The search algorithm for supercells matching can be tuned by setting its parameters directly, otherwise the default values are used.

In [ ]:
from mat3ra.made.tools.analyze.interface import ZSLInterfaceAnalyzer

zsl_analyzer = ZSLInterfaceAnalyzer(
    substrate_slab_configuration=substrate_slab_config,
    film_slab_configuration=film_slab_config,
    max_area=MAX_AREA,
    max_area_ratio_tol=MAX_AREA_TOLERANCE,
    max_length_tol=MAX_LENGTH_TOLERANCE,
    max_angle_tol=MAX_ANGLE_TOLERANCE,
    reduce_result_cell=False  # Reduces supercell matrices in analyzer
)

### 2.7. Generate interfaces with strain matcher
Interfaces are sorted by size and strain.

In [ ]:
matches = zsl_analyzer.zsl_match_holders

### 2.8. Plot interfaces by size and strain


In [ ]:
from mat3ra.notebooks_utils.ipython.entity.material.plot import plot_strain_vs_area

PLOT_SETTINGS = {
    "HEIGHT": 600,
    "X_SCALE": "log",  # or linear
    "Y_SCALE": "log",  # or linear
}

plot_strain_vs_area(matches, PLOT_SETTINGS)

### 2.9. Select the interface

Select the index for the interface with the lowest strain and the smallest number of atoms.

In [ ]:
selected_index = 0

## 3. Create the interface

In [ ]:
from mat3ra.made.tools.modify import translate_to_z_level

interface = create_zsl_interface_between_slabs(
    substrate_slab=substrate_slab,
    film_slab=film_slab,
    gap=INTERFACE_DISTANCE,
    vacuum=INTERFACE_VACUUM,
    match_id=selected_index,
    max_area=MAX_AREA,
    max_area_ratio_tol=MAX_AREA_TOLERANCE,
    max_length_tol=MAX_LENGTH_TOLERANCE,
    max_angle_tol=MAX_ANGLE_TOLERANCE,
    reduce_result_cell_to_primitive=REDUCE_RESULT_CELL_TO_PRIMITIVE,
)
interface = translate_to_z_level(interface, "center") if CENTER_IN_Z else interface

## 4. Preview the selected material

In [ ]:
visualize(interface, repetitions=[1, 1, 1])
visualize(interface, repetitions=[1, 1, 1], rotation="-90x")

## 5. Pass data to the outside runtime

In [ ]:
from mat3ra.notebooks_utils.material import set_materials

set_materials(interface)